In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

In [5]:
data = pd.read_csv('axbench_train_concept16k.csv')
data = data[
    (data.concept_genre == 'text') &
    (~data.concept_id.isin(range(10)))
]
len(data)


1538640

In [6]:
a = data["output_concept"].dropna().unique().tolist()

with open("gemma_output_concept.json", "w") as f:
    json.dump(a, f, ensure_ascii=False, indent=2)

In [11]:
prompts, rejected, accepted, steering_concepts = [], [], [], []

In [12]:
for input_prompt, input_group in tqdm(data.groupby('input'), total=len(data.input.unique())):
    negative = input_group[input_group.category == 'negative']
    if len(negative) > 0:
        for output_concept, output_group in input_group.groupby('output_concept'):
            positive = output_group[output_group.category == 'positive']
            if len(positive) > 0:
                temp_prompt = 'question:\n' + input_prompt + '\n' + 'steering concept:\n' + output_concept
                temp_accepted = positive.iloc[0]['output']
                temp_rejected = input_group[input_group.category == 'negative'].sample(n=1, random_state=44)['output'].iloc[0]
                # print(type(temp_rejected))
                if len(temp_prompt)<10:continue
                if len(temp_accepted)<10:continue
                if len(temp_rejected)<10:continue
                prompts.append(temp_prompt)
                rejected.append(temp_rejected)
                accepted.append(temp_accepted)
                steering_concepts.append(output_concept)

print(len(prompts), len(rejected), len(accepted), len(steering_concepts))

100%|██████████| 996/996 [00:59<00:00, 16.60it/s]

220090 220090 220090 220090


In [13]:
print(prompts[0])
print('__')
print(rejected[0])
print('__')
print(accepted[0])
print('__')



question:
Can whales and dolphins breathe underwater?
steering concept:
Boolean indicators of truth values
__
No, whales and dolphins cannot breathe underwater. 

They are mammals, just like humans, and need to breathe air to survive. They have lungs and must come to the surface of the water to take breaths. 

Here are some interesting facts about their breathing:

* **Blowholes:** Whales and dolphins have blowholes on top of their heads, which they use to breathe.
* **Holding their breath:** They can hold their breath for a surprisingly long time, depending on the species and their activity level.
* **Conscious breathing:** Unlike fish, whales and dolphins consciously control their breathing.


Let me know if you have any other questions about whales and dolphins! 🐬🐳

__
Whales and dolphins do not possess the ability to breathe underwater, which yields a false outcome when evaluated against the criterion of underwater respiration. The premise can be marked as 'False' to indicate the i

In [14]:
# Split by steering concept only: each unique output_concept appears in exactly one of train/valid/test (75/10/15).
SPLIT_RANDOM_SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.75, 0.10, 0.15

unique_concepts = list(dict.fromkeys(steering_concepts))
rng = np.random.default_rng(SPLIT_RANDOM_SEED)
order = rng.permutation(len(unique_concepts))
shuffled = [unique_concepts[i] for i in order]

n_concepts = len(shuffled)
n_train = int(TRAIN_FRAC * n_concepts)
n_val = int(VAL_FRAC * n_concepts)
# Assign any remainder from rounding to test so all concepts are covered once
n_test = n_concepts - n_train - n_val

train_concepts = set(shuffled[:n_train])
val_concepts = set(shuffled[n_train : n_train + n_val])
test_concepts = set(shuffled[n_train + n_val :])

assert train_concepts.isdisjoint(val_concepts)
assert train_concepts.isdisjoint(test_concepts)
assert val_concepts.isdisjoint(test_concepts)
assert train_concepts | val_concepts | test_concepts == set(unique_concepts)

def make_record(p, c, r):
    return {
        "prompt": [{"role": "user", "content": p}],
        "chosen": [{"role": "assistant", "content": c}],
        "rejected": [{"role": "assistant", "content": r}],
    }

records_train, records_val, records_test = [], [], []
for p, c, r, concept in zip(prompts, accepted, rejected, steering_concepts):
    rec = make_record(p, c, r)
    if concept in train_concepts:
        records_train.append(rec)
    elif concept in val_concepts:
        records_val.append(rec)
    else:
        records_test.append(rec)

print(
    f"Concepts: train={len(train_concepts):,} valid={len(val_concepts):,} test={len(test_concepts):,} "
    f"(total unique={n_concepts:,})"
)
print(
    f"Examples: train={len(records_train):,} valid={len(records_val):,} test={len(records_test):,}"
)

for split_name, recs, fname in [
    ("train", records_train, "DPO_data_train.jsonl"),
    ("valid", records_val, "DPO_data_valid.jsonl"),
    ("test", records_test, "DPO_data_test.jsonl"),
]:
    with open(fname, "w", encoding="utf-8") as f:
        for ex in recs:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Wrote {len(recs):,} examples to {fname}")


Concepts: train=15,771 valid=2,102 test=3,155 (total unique=21,028)
Examples: train=164,845 valid=22,127 test=33,118
Wrote 164,845 examples to DPO_data_train.jsonl
Wrote 22,127 examples to DPO_data_valid.jsonl
Wrote 33,118 examples to DPO_data_test.jsonl


In [15]:
# Quick sanity-check: inspect one train example (index 10k if available)
idx = 10_000 if len(records_train) > 10_000 else 0
print(records_train[idx] if records_train else "no train records")


{'prompt': [{'role': 'user', 'content': 'question:\nDo archaeologists dig up dinosaurs?\nsteering concept:\nterms related to articles or documents and their identifiers'}], 'chosen': [{'role': 'assistant', 'content': 'Archaeologists primarily focus on excavating sites linked to human activity, analyzing artifacts and contextual data rather than paleontological remnants. In this realm, the identification and cataloging processes resemble those of managing academic papers, where each find could be likened to a publication, complete with unique identifiers like accession numbers and metadata. Thus, while'}], 'rejected': [{'role': 'assistant', 'content': "That's a great question! \n\nArchaeologists study human history and culture, so they don't dig up dinosaurs. \n\n**Paleontologists** are the scientists who dig up dinosaur fossils! \n\nThey study these fossils to learn about dinosaurs and how they lived millions of years ago. \n"}]}


In [8]:
import json
with open("gemma_output_concept.json", "r") as f:
    gemma = set(json.load(f))
with open("llama_output_concept.json", "r") as f:
    llama = set(json.load(f))
shared = gemma & llama
print(shared)
print("gemma unique:", len(gemma))
print("llama unique:", len(llama))
print("shared:", len(shared))

{'punctuation marks and formatting elements in the text', 'expressions of sympathy and condolences', 'punctuation and formatting markers in the text', 'references to dates and times', 'references to scientific journals and articles', 'expressions of gratitude and appreciation', 'instances of the word "from."', 'phrases indicating exceptions or exclusions', 'punctuation marks and numerical values', 'names of individuals', 'mentions of government officials and their titles', 'phrases indicating possession or ownership', 'expressions of personal experiences and emotions', 'instances of the word "this" in various contexts', 'occurrences of the word "in."', 'legal terminology and references related to court proceedings', 'punctuation marks, particularly commas', 'references to religious organizations and figures', 'phrases indicating necessity or requirement', 'proper nouns, particularly names of individuals and organizations', 'prepositions and their usage in context', 'requests for assist